# Semantic Ranking

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# Load data
data_path_processed = '../data/processed/arxiv_papers.csv'

data = pd.read_csv(data_path_processed)
print(f"Shape of data: {data.shape}")
data.head()

Shape of data: (1000, 6)


,id,title,summary,category,published,combined
0,2012.11510v1,Design Rule Checking with a CNN Based Feature ...,Design rule checking (DRC) is getting increasi...,cs.LG,2020,design rule checking with a cnn based feature ...
1,2012.11638v1,Unsupervised in-distribution anomaly detection...,Anomaly detection is a key application of mach...,cs.LG|hep-ex|physics.data-an,2020,unsupervised in-distribution anomaly detection...
2,2012.11325v1,Detecting Botnet Attacks in IoT Environments: ...,The increased reliance on the Internet and the...,cs.CR|cs.LG|cs.NI,2020,detecting botnet attacks in iot environments: ...
3,2012.11327v1,Collaborative residual learners for automatic ...,Clinical coding is an administrative process t...,cs.IR|cs.LG,2020,collaborative residual learners for automatic ...
4,2012.11333v1,Ensemble model for pre-discharge icd10 coding ...,The translation of medical diagnosis to clinic...,cs.IR|cs.LG,2020,ensemble model for pre-discharge icd10 coding ...


## TF-IDF Baseline

In [5]:
# TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform(data['combined']) # Creates vocabulary and transforms it into a sparse matrix

In [6]:
# Example query vectorization
query = "self-supervised learning for image representation"
query_vector = vectorizer.transform([query])

In [7]:
# Compute cosine similarity between query and documents
tfidf_scores = cosine_similarity(query_vector, tfidf_matrix)[0]
data['tfidf_score'] = tfidf_scores

In [8]:
data_sorted = data.sort_values(by='tfidf_score', ascending=False)
# show title and score of top 5 results

with pd.option_context('display.max_colwidth', None):
    display(data_sorted[['title', 'tfidf_score']].head(5))

,title,tfidf_score
966,CompRess: Self-Supervised Learning by Compressing Representations,0.505440
384,Look Listen and Attend: Co-Attention Network for Self-Supervised Audio-Visual Representation Learning,0.248077
510,Uncovering the structure of clinical EEG signals with self-supervised learning,0.229153
814,SLM: Learning a Discourse Language Representation with Sentence Unshuffling,0.207946
93,Self-supervised Body Image Acquisition Using a Deep Neural Network for Sensorimotor Prediction,0.204004


## SBERT Embedding Similarity

In [ ]:
# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# Encode entire data
embeddings = model.encode(data['combined'].tolist(), batch_size = 32, show_progress_bar=True)
embeddings.shape

In [11]:
# Save embeddings for reuseability
path_embeddings = '../data/processed/arxiv_embeddings.npy'
np.save(path_embeddings, embeddings)

In [12]:
# Load embeddings
embeddings = np.load("../data/processed/arxiv_embeddings.npy")
embeddings.shape

(1000, 384)

In [13]:
# Category-based scoring
def category_score(categories, preferred_category):
    for i in range(len(categories)):
        category_list = categories[i].split("|")
        if preferred_category in category_list:
                return 1 / len(category_list)
    return 0

In [18]:
# Retrieve top k results for a query
def display_top_k(query, preferred_category, df, embeddings, model, k=5):
    query_embedding = model.encode([query]) # query embedding
    scores = cosine_similarity(query_embedding, embeddings)[0]
    
    df_temp = df.copy()
    df_temp['sbert_score'] = scores
    
    df_temp['final_score'] = 0.9 * df_temp['sbert_score'] + 0.1 * category_score(df_temp['category'], preferred_category)

    with pd.option_context('display.max_colwidth', None):
        display(df_temp.sort_values(by='final_score', ascending=False)[['title', 'category', 'tfidf_score', 'sbert_score', 'final_score']].head(k))

In [19]:
display_top_k("self-supervised learning for image representation", "cs.LG", data, embeddings, model)

,title,category,tfidf_score,sbert_score,final_score
984,A Framework For Contrastive Self-Supervised Learning And Designing A New Approach,cs.CV|cs.LG,0.177963,0.576773,0.619096
966,CompRess: Self-Supervised Learning by Compressing Representations,cs.CV|cs.LG,0.505440,0.563974,0.607577
94,Learning Representations by Maximizing Mutual Information Across Views,cs.LG|stat.ML,0.191488,0.550030,0.595027
262,Functional Regularization for Representation Learning: A Unified Theoretical Perspective,cs.LG|stat.ML,0.168209,0.543347,0.589012
111,Information Competing Process for Learning Diversified Representations,cs.LG|cs.CV|stat.ML,0.189759,0.491990,0.542791
